# Assignment 06 · Notebook 03: PyTorch trên AMZN (hồi quy)

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Bốn mô hình Simple RNN, LSTM, GRU, BiLSTM cùng một lớp `PyTorchRNN`, dự báo **log-return** của phiên kế tiếp
từ cửa sổ 30 phiên × 8 đặc trưng, rồi quy đổi ra giá USD: $\hat P_{t+1} = P_t\,e^{\hat r_{t+1}}$.
Mọi chỉ số so với mốc ngây thơ $\hat P_{t+1} = P_t$.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import torch
from a06.bench import eval_stock, naive_stock, save_run
from a06.data_stock import prepare_stock
from a06.export import save_metrics, write_dat
from a06.models_torch import CELLS, CELL_LABELS
from a06.train_torch import CFG, pick_device, run_torch

torch.set_num_threads(16)   # nửa còn lại của 32 luồng dành cho Keras chạy song song
print("PyTorch", torch.__version__, "· CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

PyTorch 2.13.0+cu126 · CUDA: True NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
S = prepare_stock()
for k in ("train", "val", "test"):
    d = S["dates"][k]
    print(f"{k:5s} {S['X'][k].shape}  {d[0].date()} → {d[-1].date()}  giá {S['p_next'][k].min():7.2f} – {S['p_next'][k].max():7.2f} USD")

train (4764, 30, 8)  1997-09-08 → 2016-08-11  giá    0.15 –   38.56 USD
val   (1020, 30, 8)  2016-08-12 → 2020-08-31  giá   35.95 –  172.55 USD
test  (1022, 30, 8)  2020-09-01 → 2024-09-24  giá   81.82 –  200.00 USD


## Chọn thiết bị bằng đo

Tập AMZN nhỏ (khoảng 4,8 nghìn cửa sổ), nên không mặc định GPU: đo một epoch LSTM trên CPU và GPU (sau một lượt
làm nóng), lấy bên nhanh hơn.

In [3]:
device, probe = pick_device(S, CFG["amzn"])
print("s/epoch:", {k: round(v, 3) for k, v in probe.items()}, "→ dùng", device)

s/epoch: {'cpu': 0.652, 'cuda': 0.223} → dùng cuda


## Huấn luyện

AdamW (lr 1e-3, weight decay 1e-4), clip chuẩn gradient 1,0, lô 64, tối đa 60 epoch, dừng sớm khi val loss
không giảm sau 8 epoch; giữ trọng số của epoch có val loss nhỏ nhất.

In [4]:
out = run_torch("amzn", S, device)
meta = save_run("amzn", "torch", out)

amzn torch rnn    epoch tốt  7/15 val 0.2213 · 0.25 s/epoch trên cuda


amzn torch lstm   epoch tốt  5/13 val 0.2206 · 0.21 s/epoch trên cuda


amzn torch gru    epoch tốt  7/15 val 0.2199 · 0.17 s/epoch trên cuda


amzn torch bilstm epoch tốt  1/9 val 0.2226 · 0.27 s/epoch trên cuda


## Kết quả trên tập test (giá USD)

In [5]:
res = {c: eval_stock(S, out[c]["pred_test"]) for c in CELLS}
naive = naive_stock(S)
rows = {CELL_LABELS[c]: {k: res[c][k] for k in ("rmse", "mae", "r2", "r2_ret", "dir_acc")} for c in CELLS}
rows["Mốc ngây thơ"] = naive
pd.DataFrame(rows).T.round(4)

,rmse,mae,r2,r2_ret,dir_acc
Simple RNN,3.1371,2.2902,0.9880,-0.0148,0.4843
LSTM,3.1309,2.2755,0.9880,-0.0063,0.5147
GRU,3.1409,2.2858,0.9879,-0.0136,0.5059
BiLSTM,3.1288,2.2776,0.9880,-0.0019,0.5088
Mốc ngây thơ,3.1239,2.2741,0.9881,-0.0000,NaN


In [6]:
# 150 phiên cuối của test: giá thật, mốc ngây thơ và dự báo của 4 mô hình
n = 150
cols = {"i": np.arange(n), "true": S["p_next"]["test"][-n:], "naive": S["p_now"]["test"][-n:]}
cols.update({c: res[c]["price_pred"][-n:] for c in CELLS})
write_dat("amzn_torch_pred", cols)

d = S["dates"]
save_metrics("torch_amzn", {
    "device": device, "probe": probe,
    "test_start": str(d["test"][0].date()), "test_end": str(d["test"][-1].date()),
    "plot_start": str(d["test"][-n].date()), "plot_n": n,
    "naive": naive,
    "models": {c: {**{k: v for k, v in res[c].items() if k != "price_pred"},
                   **{k: meta[f"amzn_torch_{c}"][k] for k in ("params", "best_epoch", "epochs_run", "epoch_s", "train_s")}}
               for c in CELLS}})
print("đã ghi outputs/metrics/torch_amzn.json")

đã ghi outputs/metrics/torch_amzn.json
